# Week 6 — Multi-table Joins and Complex Aggregations: Three-Table Joins
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Chain three tables together in a single query — `order_items` → `products` → `product_category_translation` — to answer a business question no single table can answer
- Aggregate across a multi-table join with `COUNT(DISTINCT ...)`, `SUM`, and `AVG`, and know why `COUNT(*)` is the wrong choice once a join has fanned rows out
- Recognise which Olist tables hold more than one row per `order_id`, and read a joined result at the correct grain before you trust the numbers it reports

Run the setup cell first. It loads all 8 Olist tables into a SQLite database and connects the
`%%sql` magic to it — everything below depends on it, so run it before anything else.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

Olist's category manager wants one thing: a ranked list of product categories by revenue, so she
knows where to push next quarter's marketing budget. It sounds like a one-table question — and it
is not answerable from any one table in the database.

The money lives in `order_items` (112,650 line items, each with a `price`), but `order_items` only
records a `product_id` — it has no idea what *kind* of product that is. The category lives in
`products` (32,951 rows), but only in Portuguese: `beleza_saude`, `moveis_decoracao`,
`ferramentas_jardim`. The English name your manager actually reads lives in a third, tiny table —
`product_category_translation`, just 71 rows mapping Portuguese to English.

So the answer is spread across three tables, and **only a join can put it back together**. Week 3
taught you to join two tables; this week you chain that same idea one step further. The pattern
scales: each `JOIN` bolts one more table onto the result you already have, using a key the two
share. Three tables, two joins. Four tables, three joins. The syntax never gets harder — but the
*reasoning about row counts* does, which is where the second half of this session goes.

## 1. Chaining three tables — revenue by product category

A three-table join is not a new piece of syntax. It is the two-table join you already know, written
twice: SQLite joins the first two tables into one intermediate result, then joins the third table
onto *that* result. You read it top to bottom as a chain, and each `JOIN ... ON ...` line names the
one key that links the new table to what came before it.

Here the chain is `order_items → products → product_category_translation`. The first link joins on
`product_id`, which both `order_items` and `products` carry. The second link joins on
`product_category_name` — the Portuguese category string that `products` stores and
`product_category_translation` translates. Notice that `order_items` and
`product_category_translation` share **no** column at all: you can only reach the English name by
travelling through `products` in the middle. That is the essence of a multi-table join — following
a path through the schema, one shared key at a time.

Before aggregating anything, it's worth seeing what the joined rows actually look like. The query
below does no `GROUP BY` — it just shows five joined line items so you can watch the chain work:
the price from `order_items`, the Portuguese name from `products`, and the English name from the
translation table, all lined up on one row.

In [ ]:
%%sql
-- Look at the raw joined rows FIRST, before any aggregation.
-- Each row = one line item, carrying columns from all three tables.
SELECT oi.order_id,
       p.product_category_name           AS category_portuguese,
       t.product_category_name_english   AS category_english,
       oi.price
FROM order_items oi
JOIN products p                     ON oi.product_id = p.product_id
JOIN product_category_translation t ON p.product_category_name = t.product_category_name
LIMIT 5

That is the whole trick: after two `JOIN`s, every row carries columns from all three tables, and you
can `SELECT`, filter, and aggregate them as if they had always lived together.

Now aggregate. `GROUP BY category` collapses those joined rows into one row per English category
name, and four aggregates describe each category: how many distinct orders it appeared in, how many
distinct sellers offer it, its total revenue, and its average item price. Note
`COUNT(DISTINCT oi.order_id)` rather than `COUNT(*)` — an order containing three health & beauty
items produces three joined rows, and counting rows would count that single order three times. The
reason that matters gets a full treatment in the "Going deeper" section below.

In [ ]:
%%sql
-- Business question: what is the revenue by product category (in English)?
SELECT t.product_category_name_english AS category,
       COUNT(DISTINCT oi.order_id)     AS order_count,
       COUNT(DISTINCT oi.seller_id)    AS seller_count,
       ROUND(SUM(oi.price), 2)         AS total_revenue,
       ROUND(AVG(oi.price), 2)         AS avg_price
FROM order_items oi
JOIN products p                     ON oi.product_id = p.product_id
JOIN product_category_translation t ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10
-- Expected top row: health_beauty | 8,836 orders | 492 sellers | R$1,258,681.34 | avg R$130.16

**Expected output — the top 10 categories by revenue:**

| category | order_count | seller_count | total_revenue | avg_price |
|---|---|---|---|---|
| health_beauty | 8,836 | 492 | 1,258,681.34 | 130.16 |
| watches_gifts | 5,624 | 101 | 1,205,005.68 | 201.14 |
| bed_bath_table | 9,417 | 196 | 1,036,988.68 | 93.30 |
| sports_leisure | 7,720 | 481 | 988,048.97 | 114.34 |
| computers_accessories | 6,689 | 287 | 911,954.32 | 116.51 |
| furniture_decor | 6,449 | 370 | 729,762.49 | 87.56 |
| cool_stuff | 3,632 | 267 | 635,290.85 | 167.36 |
| housewares | 5,884 | 468 | 632,248.66 | 90.79 |
| auto | 3,897 | 383 | 592,720.11 | 139.96 |
| garden_tools | 3,518 | 237 | 485,256.46 | 111.63 |

Read it as a business person, not just as a SQL result. `health_beauty` leads on revenue
(R$1,258,681.34) with 8,836 orders spread across 492 different sellers — a broad, competitive
category. `watches_gifts` is only R$53,675 behind it on revenue, but gets there with **5,624**
orders from just **101** sellers, at an average item price of R$201.14 — the highest in the top ten.
Two completely different business shapes, and one query surfaced both. Meanwhile `bed_bath_table`
has the *most* orders of anyone (9,417) yet ranks only third on revenue, because its average item
sells for R$93.30. Volume and value are not the same story, and the category manager needs both.

## 2. Joining on `order_id` — does price differ by review score?

The second business question changes the join key rather than the number of tables. Customer
satisfaction lives in `order_reviews` (a 1–5 `review_score` per review), and prices live in
`order_items`. Neither table knows anything about the other — but both carry `order_id`, so that
column is the bridge between them.

The question itself is a genuinely useful one: *do unhappy customers tend to have bought more
expensive items?* If they do, that points at expectation — people who spend more expect more, and
punish harder when delivery disappoints. If they don't, dissatisfaction is being driven by something
other than price, and the operations team should look at delivery times instead. Either way, this is
a question that only a join can answer, because no single table holds both the price and the score.

In [ ]:
%%sql
-- Business question: does price differ by review score?
SELECT r.review_score,
       COUNT(DISTINCT oi.order_id) AS order_count,
       ROUND(AVG(oi.price), 2)     AS avg_item_price,
       ROUND(SUM(oi.price), 2)     AS total_revenue
FROM order_items oi
JOIN order_reviews r ON oi.order_id = r.order_id
GROUP BY r.review_score
ORDER BY r.review_score
-- Expected: 5 rows, one per score. Score 1 = 10,854 orders, avg R$127.35;
--           score 5 = 57,006 orders, avg R$121.22

**Expected output:**

| review_score | order_count | avg_item_price | total_revenue |
|---|---|---|---|
| 1 | 10,854 | 127.35 | 1,812,828.22 |
| 2 | 3,086 | 115.85 | 448,799.56 |
| 3 | 8,107 | 110.06 | 1,037,092.59 |
| 4 | 19,065 | 118.60 | 2,528,015.01 |
| 5 | 57,006 | 121.22 | 7,700,489.39 |

> **Discussion — 5 minutes, talk it out.** Items in 1-star orders are actually *slightly more
> expensive* on average (R$127.35) than items in 5-star orders (R$121.22). Does price drive
> dissatisfaction? What else might explain this — do pricier items ship from further away, take
> longer to arrive, or simply carry higher expectations? Notice too how lopsided the volumes are:
> 57,006 orders scored 5 versus 10,854 scored 1. Olist's customers are mostly happy; the interesting
> question is what separates the unhappy minority. Note also that the relationship is not a clean
> slope — score 3 has the *lowest* average price of all (R$110.06), which is a good reminder that
> "1-star is dearer than 5-star" is a much weaker finding than a straight line would be.

---
## 🤖 Using DeepSeek this week

You've had DeepSeek available since Week 4, and multi-table joins are exactly the kind of task it's
good at drafting: you describe the tables and the business question, it writes the join chain. The
rule has not changed though — **draft, run, verify, then trust.**

Joins are where AI-drafted SQL fails in the most dangerous way: not with an error, but with a number
that looks perfectly reasonable and is quietly wrong. The two failure modes to watch for are (a) it
joins on the wrong key or skips the middle table entirely, and (b) it writes `COUNT(*)` where the
join has fanned rows out, inflating every count. Neither one crashes. Both give you a plausible
answer you'd have no reason to doubt.

The protocol:
1. **Ask** DeepSeek: *"Using SQLite, write a query that returns the total revenue for the
   `health_beauty` category, joining `order_items` to `products` on `product_id` and
   `products` to `product_category_translation` on `product_category_name`."*
2. **Run** whatever it gives you — in a `%%sql` cell, against this database.
3. **Verify** against a number you already know is right. You computed `health_beauty` revenue two
   sections ago: R$1,258,681.34. If the AI's query returns anything else, its query is wrong — not
   the data. Only once it reproduces a value you've already verified should you trust it on a
   question you *haven't* solved yourself.

In [ ]:
%%sql
-- Step 3 of the protocol: a focused check on a value we already verified above.
-- If a DeepSeek-drafted join returns anything other than this, the query is wrong.
SELECT ROUND(SUM(oi.price), 2) AS health_beauty_revenue
FROM order_items oi
JOIN products p                     ON oi.product_id = p.product_id
JOIN product_category_translation t ON p.product_category_name = t.product_category_name
WHERE t.product_category_name_english = 'health_beauty'   -- Expected: 1,258,681.34

## Going deeper — join fan-out, and why `COUNT(DISTINCT ...)` keeps showing up

Every join question comes down to one word: **grain**. The grain of a table is what one row of it
represents. `orders` has a grain of one row per order — 99,441 rows, 99,441 distinct `order_id`s.
But three Olist tables do **not** have that grain:

| Table | Rows | Distinct `order_id` | Grain |
|---|---|---|---|
| `orders` | 99,441 | 99,441 | one row per order — safe base |
| `order_items` | 112,650 | 98,666 | **many rows per order** (one per line item) |
| `order_payments` | 103,886 | 99,440 | **many rows per order** (instalments, vouchers) |
| `order_reviews` | 99,224 | 98,673 | **many rows per order** (some orders reviewed twice) |

When you join a many-row-per-order table, each order's row gets *repeated* once per matching row on
the other side. That repetition is called **fan-out**, and it silently multiplies anything you then
count or sum. This is why Concept 1 wrote `COUNT(DISTINCT oi.order_id)` and not `COUNT(*)`: an order
with three health & beauty items contributes three rows to the join, and `COUNT(*)` would report it
as three orders.

The rule to carry forward: **never `SUM`, `AVG`, or `COUNT(*)` across a query that directly joins two
or more of `order_items` / `order_payments` / `order_reviews`.** Collapse the extra table to one row
per `order_id` in a `WITH` CTE first, then join that. Concept 2 above joins `order_items` straight to
`order_reviews` — two fan-out tables — which is fine for *comparing* scores as the curriculum does,
but you should know the caveat: those 547 twice-reviewed orders get counted under both scores, and
`total_revenue` per score is inflated for exactly those orders. When the number has to be exact,
pre-aggregate first. Run the cell below to see the grain for yourself.

Check the grain **before** you join, one table at a time. Compare `COUNT(*)` (how many rows the
table has) against `COUNT(DISTINCT order_id)` (how many orders it describes). If the two numbers
match, the table is one row per order and is safe to join. If they differ, the table will fan out.

Start with `orders`, the safe base.

In [ ]:
%%sql
-- Grain check 1 of 4 — orders (the safe base table)
SELECT COUNT(*) AS row_count, COUNT(DISTINCT order_id) AS distinct_orders
FROM orders
-- Expected: 99,441 rows / 99,441 distinct orders — identical, so one row per order

99,441 and 99,441 — identical, so `orders` really is one row per order. Now `order_items`.

In [ ]:
%%sql
-- Grain check 2 of 4 — order_items (one row per line item, not per order)
SELECT COUNT(*) AS row_count, COUNT(DISTINCT order_id) AS distinct_orders
FROM order_items
-- Expected: 112,650 rows / 98,666 distinct orders — 13,984 extra rows of fan-out

112,650 rows describing only 98,666 orders: roughly 14,000 rows of fan-out, caused by orders that
contain more than one line item. Next, `order_payments`.

In [ ]:
%%sql
-- Grain check 3 of 4 — order_payments (instalments and vouchers add rows)
SELECT COUNT(*) AS row_count, COUNT(DISTINCT order_id) AS distinct_orders
FROM order_payments
-- Expected: 103,886 rows / 99,440 distinct orders

103,886 rows against 99,440 orders — an order paid in instalments, or part-paid with a voucher,
contributes several payment rows. Finally, `order_reviews`.

In [ ]:
%%sql
-- Grain check 4 of 4 — order_reviews (a few orders were reviewed more than once)
SELECT COUNT(*) AS row_count, COUNT(DISTINCT order_id) AS distinct_orders
FROM order_reviews
-- Expected: 99,224 rows / 98,673 distinct orders

99,224 rows for 98,673 orders — a small fan-out, but still enough to inflate a `SUM` or double-count
an order. Three of the four tables fan out, and only `orders` does not. That is exactly why every
count in this session is written as `COUNT(DISTINCT oi.order_id)`.

Here is the safe pattern in full, and a preview of Thursday's work. To put **revenue and average
review score side by side per category**, the `rev` CTE first collapses `order_reviews` to exactly
one row per `order_id`. Only then is it joined on — now a clean 1:1 link that cannot fan anything
out. Revenue still comes from `order_items` at line-item grain, which is correct, because revenue
*is* a per-item quantity.

In [ ]:
%%sql
-- The SAFE pattern: collapse the fan-out table to 1 row per order_id FIRST, then join.
WITH rev AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
)
SELECT t.product_category_name_english AS category,
       COUNT(DISTINCT oi.order_id)     AS order_count,
       ROUND(SUM(oi.price), 2)         AS total_revenue,
       ROUND(AVG(r.review_score), 2)   AS avg_review
FROM order_items oi
JOIN products p                     ON oi.product_id = p.product_id
JOIN product_category_translation t ON p.product_category_name = t.product_category_name
LEFT JOIN rev r                     ON oi.order_id = r.order_id   -- 1:1 now — no fan-out
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 5
-- Expected: health_beauty still R$1,258,681.34 over 8,836 orders — the CTE protected the total

## Common mistakes

**Mistake — using `COUNT(*)` as "the number of orders" after joining `order_items`.** It is the most
natural thing in the world to write, it never errors, and it is wrong. After the join, one row is one
*line item*, not one order, so `COUNT(*)` answers a question nobody asked. For `health_beauty` the
gap is 9,670 rows against 8,836 real orders — a 9% overstatement that would quietly walk straight
into a management report.

**Mistake — forgetting the middle table.** `order_items` and `product_category_translation` share no
column, so `JOIN product_category_translation t ON oi.product_id = t.product_category_name` is
nonsense: SQLite will happily run it and return zero rows, because a product UUID never equals a
category name. You must travel through `products`.

**Mistake — an unqualified column name.** Once three tables are in play, writing bare
`product_category_name` is ambiguous — both `products` and `product_category_translation` have a
column by that name — and SQLite raises an "ambiguous column name" error. Always alias your tables
(`oi`, `p`, `t`) and prefix every column. It also makes the query far easier to read six months
later.

The cell below shows the count mistake as a comment, then runs the correct version live so you can
see both numbers the query would have produced.

In [ ]:
%%sql
-- ── COMMON MISTAKE ──────────────────────────────────────────────────
-- WRONG — after joining order_items, one row is one LINE ITEM, not one order.
-- COUNT(*) counts fanned-out rows and overstates the order count:
--   SELECT t.product_category_name_english, COUNT(*) AS order_count ...
-- CORRECT — COUNT(DISTINCT oi.order_id) counts each order once.
-- Both are shown side by side here so you can see the size of the gap:
SELECT t.product_category_name_english AS category,
       COUNT(*)                        AS joined_rows,     -- Expected: 9,670  (WRONG as an order count)
       COUNT(DISTINCT oi.order_id)     AS order_count      -- Expected: 8,836  (CORRECT)
FROM order_items oi
JOIN products p                     ON oi.product_id = p.product_id
JOIN product_category_translation t ON p.product_category_name = t.product_category_name
WHERE t.product_category_name_english = 'health_beauty'
GROUP BY category

## Group Exercise — trace the chain, then predict the fan-out

⏱ ~8 min · pairs or threes · discussion only, no code required

Work through both of today's concepts out loud with the person next to you. Nominate one person to
report back one sentence per part.

**Part A — chain a new path (Concept 1).** Your manager now asks: *"Which product categories are the
sellers in São Paulo state actually shipping?"* Seller state lives in `sellers` (`seller_state`), the
English category name lives in `product_category_translation`, and neither table shares a column with
the other. Between you, name the tables you would put in the chain, in order, and the key each
`JOIN ... ON ...` would use. How many `JOIN` lines does your chain need?

**Part B — predict the damage (Concept 2).** Someone on your team writes
`SELECT COUNT(*) FROM orders o JOIN order_payments p ON o.order_id = p.order_id` and reports the
result as "the number of orders". Using the grain table above, decide together: will that number be
too high, too low, or exactly right — and roughly by how much? What one word would you change to fix
it?

Be ready to defend your answer to Part B with a number, not just a hunch.

## Mini-challenge — your turn

⏱ ~5–10 min

Take the Concept 1 three-table join and point it at a single category instead of ranking all of
them. Write a query that returns the `order_count`, `seller_count`, `total_revenue`, and `avg_price`
for **`garden_tools` only**.

Two hints:
- Keep all three tables and both `JOIN ... ON ...` lines exactly as they are — the chain doesn't
  change, only the filter does.
- Add `WHERE t.product_category_name_english = 'garden_tools'` before the `GROUP BY`, and drop the
  `ORDER BY` / `LIMIT` (there is only one row to return).

**Expected:** one row — 3,518 orders, 237 sellers, R$485,256.46 total revenue, R$111.63 average
price. Check it against the top-10 table above: your single row should match the `garden_tools` line
exactly.

**Stretch question to discuss:** looking at that top-10 table, which category has the *highest*
average item price — and is it the same one that earns the most revenue? (Answer: `watches_gifts` at
R$201.14 average, while `health_beauty` at R$130.16 leads on total revenue. High price per item and
high total revenue are two different kinds of winner.)

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Pattern | What it does | Example |
|---|---|---|
| Chained `JOIN` | bolts a third table onto the result of the first join, one shared key at a time | `FROM order_items oi JOIN products p ON oi.product_id = p.product_id JOIN product_category_translation t ON p.product_category_name = t.product_category_name` |
| Table aliases | keeps three-table queries readable and avoids "ambiguous column name" | `order_items oi`, `products p`, `product_category_translation t` |
| `COUNT(DISTINCT ...)` | counts real orders, not fanned-out join rows | `COUNT(DISTINCT oi.order_id) AS order_count` |
| `GROUP BY` over a join | aggregates columns that came from different tables | `GROUP BY t.product_category_name_english` |
| Pre-aggregating CTE | collapses a fan-out table to 1 row per `order_id` before joining | `WITH rev AS (SELECT order_id, AVG(review_score) AS review_score FROM order_reviews GROUP BY order_id)` |

**What you can now answer that you couldn't yesterday:** revenue by English category name
(`health_beauty` leads at R$1,258,681.34), and whether price tracks satisfaction (barely — 1-star
orders average R$127.35 per item against 5-star's R$121.22).

---
**Coming up Thursday**: **geographic revenue analysis** — a full-pipeline join across
`orders → customers → order_payments`, filtered to delivered orders and grouped by
`customer_state`, so you can see total payment value, average order value, *and* average delivery
days per state in one result. You'll compute delivery duration from two TEXT timestamps with
`julianday()`, and the exercises push the same chain out to four and five tables.